In [2]:
import ffsim
import numpy as np
from ffsim.variational.util import interaction_pairs_spin_balanced
from las_lucj.ucj_frag import ucj_frag_circuit
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

GATE_MAP = {
    ffsim.UCJOpSpinBalanced: ffsim.qiskit.UCJOpSpinBalancedJW,
    ffsim.UCJOpSpinUnbalanced: ffsim.qiskit.UCJOpSpinUnbalancedJW,
}


def identity_ucj_op_spin_unbalanced(
    norb: int,
    *,
    n_reps: int = 1,
    with_final_orbital_rotation: bool = False,
) -> ffsim.UCJOpSpinUnbalanced:
    diag_coulomb_mats = np.zeros((n_reps, 3, norb, norb))
    orbital_rotations = np.stack(
        [np.stack([np.eye(norb), np.eye(norb)]) for _ in range(n_reps)]
    )
    final_orbital_rotation = None
    if with_final_orbital_rotation:
        final_orbital_rotation = np.stack([np.eye(norb), np.eye(norb)])
    return ffsim.UCJOpSpinUnbalanced(
        diag_coulomb_mats=diag_coulomb_mats,
        orbital_rotations=orbital_rotations,
        final_orbital_rotation=final_orbital_rotation,
    )

In [4]:
norb = 36
nelec = (27, 27)
n_alpha, n_beta = nelec
norb_small_1 = 18
nelec_small_1 = (14, 13)
n_alpha_small_1, n_beta_small_1 = nelec_small_1
norb_small_2 = 18
nelec_small_2 = (13, 14)
n_alpha_small_2, n_beta_small_2 = nelec_small_2
assert norb_small_1 + norb_small_2 == norb
assert n_alpha_small_1 + n_alpha_small_2 == n_alpha
assert n_beta_small_1 + n_beta_small_2 == n_beta

pairs_aa, pairs_ab = interaction_pairs_spin_balanced("heavy-hex", norb_small_1)
ucj_op_small_1 = ffsim.random.random_ucj_op_spin_unbalanced(
    norb=norb_small_1,
    n_reps=1,
    interaction_pairs=(pairs_aa, pairs_ab, pairs_aa),
)
# ucj_op_small_1 = identity_ucj_op_spin_unbalanced(norb=norb_small_1, n_reps=1)
pairs_aa, pairs_ab = interaction_pairs_spin_balanced("heavy-hex", norb_small_2)
ucj_op_small_2 = ffsim.random.random_ucj_op_spin_unbalanced(
    norb=norb_small_2,
    n_reps=1,
    interaction_pairs=(pairs_aa, pairs_ab, pairs_aa),
)
# ucj_op_small_2 = identity_ucj_op_spin_unbalanced(norb=norb_small_2, n_reps=1)
ucj_op_big = ffsim.random.random_ucj_op_spin_balanced(
    norb=norb,
    n_reps=1,
    interaction_pairs=interaction_pairs_spin_balanced("heavy-hex", norb),
)

circuit_opt = ucj_frag_circuit(
    [ucj_op_small_1, ucj_op_small_2], [nelec_small_1, nelec_small_2], ucj_op_big
)

# Create a generic backend and transpile the circuit to it
backend = GenericBackendV2(2 * norb, basis_gates=["cz", "id", "rz", "sx", "x"])
pass_manager = generate_preset_pass_manager(optimization_level=3, backend=backend)
# Set the pre-initialization stage of the pass manager with passes suggested by ffsim
pass_manager.pre_init = ffsim.qiskit.PRE_INIT
transpiled = pass_manager.run(circuit_opt)

print(transpiled.count_ops())
print(transpiled.depth(lambda instruction: instruction.operation.num_qubits == 2))

OrderedDict({'sx': 22437, 'rz': 22233, 'cz': 5836, 'x': 90})
279
